In [2]:
from pathlib import Path

import prism

from imagematerials.eol import eol_preprocess
from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    EndOfLife,
    GenericMaterials,
    GenericStocks,
    Maintenance,
    MaterialIntensities,
    RestOf
)
from imagematerials.preprocessing import get_preprocessing_data
import numpy as np

from imagematerials.rest_of import rest_of_preprocessing
import matplotlib.pyplot as plt

from imagematerials.electricity.preprocessing import (
    get_preprocessing_data_gen,
    get_preprocessing_data_grid)


In [3]:
scenario_name = "SSP2_M_CP"
scen = "SSP2"
variant = "M_CP"

In [4]:
climate_policy_scenario_dir = Path("..", "data", "raw", "image", scenario_name)
scenario_base_path = Path("../data/raw") / 'circular_economy_scenarios'

path_image_materials = Path("C:/Coding/image-materials")

raw_data = path_image_materials / "data/raw"

In [5]:
#load historic data
from imagematerials.rest_of.resource_model import ResourceModel

steel = ResourceModel(resource_group = 'metals', resource = 'steel', 
                        image_mat_available = True, start_year = 1971,
                        scenario=scenario_name,
                        convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                        trade_data=True, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image"))

aluminium = ResourceModel(resource_group = 'metals', resource = 'aluminium', 
                        image_mat_available = True, start_year = 1998, 
                        scenario=scenario_name, end_year = 2024, path_input_data="../data/raw/rest-of", 
                        path_input_data_image = Path("../data/raw/image")
                        )

cement = ResourceModel(resource_group = 'nmm', resource = 'cement', 
                    image_mat_available = True, start_year = 1971, 
                    scenario=scenario_name,
                    convert_image=True, end_year = 2012, convert_to_tons = 1/1000_000, 
                    trade_data=True, path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

copper = ResourceModel(resource_group = 'metals', resource = 'copper', 
                       image_mat_available = True, start_year = 1990,
                       scenario= scenario_name, end_year = 2011,
                       path_input_data="../data/raw/rest-of", 
                       path_input_data_image = Path("../data/raw/image"))

sand = ResourceModel(resource_group = 'nmm', resource = 'sand_gravel_crushed_rock', 
                       image_mat_available = True, start_year = 1970, scenario= scenario_name, 
                       path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

limestone = ResourceModel(resource_group = 'nmm', resource = 'limestone', 
                    image_mat_available = False, start_year = 1970, scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))

clay = ResourceModel(resource_group = 'nmm', resource = 'clays', 
                    image_mat_available = False, start_year = 1970, 
                    scenario=scenario_name, 
                    path_input_data="../data/raw/rest-of", 
                    path_input_data_image = Path("../data/raw/image"))



In [6]:


# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)


bld_sector = get_preprocessing_data("buildings", Path("..", "data", "raw"), 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None) 
vhc_sector = get_preprocessing_data("vehicles", raw_data, 
                                    climate_policy_scenario_dir, 
                                    circular_economy_scenario_dirs = None)
rest_sector = rest_of_preprocessing(raw_data, 
                    image_scenario_directory = climate_policy_scenario_dir)


# TODO fix this for real in the future
prep_data_vhc = vhc_sector.prep_data
prep_data_el = get_preprocessing_data_gen(path_image_materials, scen, variant)
_, prep_data_grid = get_preprocessing_data_grid(path_image_materials, scen, variant)


vhc_sector = Sector('vehicles', prep_data_vhc)
rest_sector = Sector(name='rest_of', 
                    data = rest_sector,)
sec_electr_gen = Sector("generation", prep_data_el)
sec_electr_grid = Sector("grid", prep_data_grid)

factory = ModelFactory(
[bld_sector, vhc_sector, rest_sector, sec_electr_gen, sec_electr_grid], complete_timeline
).add(GenericStocks, ["buildings", "vehicles", "generation", "grid"]
).add(GenericMaterials,  "vehicles"
).add(MaterialIntensities, ["buildings", "generation", "grid"]
).add(RestOf, "rest_of", input_sources={
    "gompertz_coefs": "rest_of",
    "gdp_per_capita": "rest_of",
    "population": "rest_of",
    "historic_diff_consumption": "rest_of"
}
)
model = factory.finish()

import warnings
with warnings.catch_warnings():
    warnings.filterwarnings("ignore")
    model.simulate(simulation_timeline)



c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1689: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1689: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\indexing.py:1689: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  array[key] = value


Path to image output: C:\Coding\image-materials\data\raw\SSP2_M_CP\EnergyServices
gcap_stock to xarray Dataset
gcap_types_materials to xarray Dataset
gcap_lifetime_distr not in conversion_table
grid_stock_lines to xarray Dataset
materials_grid_kgperkm to xarray Dataset
lifetime_grid_distr not in conversion_table
grid_stock_add to xarray Dataset
materials_grid_add_kgperunit to xarray Dataset
lifetime_grid_distr not in conversion_table


In [28]:
model.grid["inflow_materials"].to_array()

<xarray.DataArray (time: 101, Region: 26, Type: 6, material: 9)> Size: 1MB
<Quantity([[[[2.61038949e+06 0.00000000e+00 9.74061131e+06 ... 1.57233443e-01
    2.37422487e+04 1.15189215e+06]
   [1.22997483e+05 0.00000000e+00 1.60366939e+08 ... 0.00000000e+00
    0.00000000e+00 7.32537865e+07]
   [7.88188217e+07 0.00000000e+00 3.05519211e+07 ... 0.00000000e+00
    0.00000000e+00 2.43901891e+06]
   [5.45570020e+06 0.00000000e+00 1.12965086e+07 ... 0.00000000e+00
    0.00000000e+00 3.08086599e+07]
   [1.40481915e+06 0.00000000e+00 1.45310695e+08 ... 0.00000000e+00
    0.00000000e+00 2.07634101e+06]
   [2.92648173e+04 0.00000000e+00 6.52549682e+07 ... 1.23051585e+02
    0.00000000e+00 3.15767378e+07]]

  [[1.60724854e+07 0.00000000e+00 5.99741277e+07 ... 9.68105417e-01
    1.46183911e+05 7.09233995e+06]
   [7.57310451e+05 0.00000000e+00 9.87398737e+08 ... 0.00000000e+00
    0.00000000e+00 4.51032467e+08]
   [4.36347793e+08 0.00000000e+00 1.69138070e+08 ... 0.00000000e+00
    0.00000000e+00 1.35026190e+07]
   [3.02032267e+07 0.00000000e+00 6.25384459e+07 ... 0.00000000e+00
...
    0.00000000e+00 5.70637367e+06]
   [1.45001943e+07 0.00000000e+00 3.00239318e+07 ... 0.00000000e+00
    0.00000000e+00 8.18834503e+07]
   [3.47803055e+07 0.00000000e+00 3.59758077e+09 ... 0.00000000e+00
    0.00000000e+00 5.14057447e+07]
   [8.23070575e+05 0.00000000e+00 1.83529061e+09 ... 0.00000000e+00
    0.00000000e+00 8.88093151e+08]]

  [[0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [6.80188071e+05 0.00000000e+00 8.86844808e+08 ... 0.00000000e+00
    0.00000000e+00 4.05100581e+08]
   [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [3.26050655e+07 0.00000000e+00 6.75116651e+07 ... 0.00000000e+00
    0.00000000e+00 1.84122723e+08]
   [0.00000000e+00 0.00000000e+00 0.00000000e+00 ... 0.00000000e+00
    0.00000000e+00 0.00000000e+00]
   [3.58274282e+05 0.00000000e+00 7.98883406e+08 ... 0.00000000e+00
    0.00000000e+00 3.86577950e+08]]]], 'kilogram')>
Coordinates:
  * time      (time) int64 808B 1960 1961 1962 1963 1964 ... 2057 2058 2059 2060
  * Region    (Region) <U13 1kB 'Brazil' 'C.Europe' ... 'W.Africa' 'W.Europe'
  * Type      (Type) <U17 408B 'HV - Substations' ... 'MV - Transformers'
  * material  (material) <U9 324B 'Aluminium' 'Co' ... 'Plastics' 'Steel'

In [7]:
# sum inflow materials for steel, sum also per types keep regions and year

materials_dict_metal = {
    'Steel' : 'Steel',
    'Aluminium' : 'Aluminium',
    'Copper' : 'Cu',
}

materials_dict_nmm = {
    'Cement' : 'Cement',
    'Sand' : 'Sand'
}

# Conversion factors
# always taking the lower range numbers to be cautios

# https://civiltoday.com/civil-engineering-materials/cement/10-cement-ingredients-with-functions
# Cement: Lime 60-75%, Silica 17-25%, other aggregates
# https://concretesupplyco.com/concrete-basics/
# Concrete:  10% cement, 20% air and water, 30% sand, and 40% gravel --> 30% + 40% = 70%
# https://samsa.org.uk/key_uses/glass.php, https://www.carmeuse.com/na-en/references/case-studies-success-stories/limestone-glassmaking-what-you-need-know
# Sand (silica) in glass: 70%, lime: 15%

from imagematerials.rest_of.preprocessing.sum_sectors_image_materials import sum_inflows_for_output

total_material_dict_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals', save = True)
total_material_dict_nmm = sum_inflows_for_output(model, materials_dict_nmm, 'nmm', save = True)



Steel
done steel
Aluminium
done aluminium
Copper
done copper
Cement


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: Uni

done cement
Sand
done sand_gravel_crushed_rock


In [ ]:

from imagematerials.rest_of.const import REGION_TO_CLASS_DICT_IMAGE_MAT_NR

def sum_inflows_for_output(model_name, materials_dict, resource_group, save = True):
    cement_in_concrete = 0.1
    sand_in_cement_conversion = 0.17 #(silica)
    sand_gravel_in_concrete_conversion = 0.7
    sand_in_glass_conversion = 0.7

    only_buildings = ['Cement', 'Concrete']
    only_vehicles = ['Glass']
    not_in_any = ['Sand']
    total_material_dict = {}

    # regions electricity and generation
    inflow_materials_grid = model_name.grid["inflow_materials"].to_array()
    inflow_materials_generation = model_name.generation["inflow_materials"].to_array()

     # replace region names to numbers
    # replace region names to numbers in the 'Region' coordinate
    inflow_materials_grid = inflow_materials_grid.assign_coords(
        Region = inflow_materials_grid.coords["Region"].to_series().map(REGION_TO_CLASS_DICT_IMAGE_MAT_NR)
    )
    inflow_materials_generation = inflow_materials_generation.assign_coords(
        Region = inflow_materials_generation.coords["Region"].to_series().map(REGION_TO_CLASS_DICT_IMAGE_MAT_NR)
    )

    for key, value in materials_dict.items():
        if key not in only_buildings and key not in only_vehicles and not key in not_in_any:
            inflow_buildings = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material=key).loc[1961:]
            inflow_vehicles = model_name.vehicles.get('inflow_materials').to_array().sum(['Type']).sel(material=value).loc[1961:]
            inflow_electricity_genereation = inflow_materials_generation.sum(['Type']).sel(material=value).loc[1961:]
            inflow_electricity_grid = inflow_materials_grid.sum(['Type']).sel(material=value).loc[1961:]
            total_material = inflow_vehicles + inflow_electricity_genereation + inflow_electricity_grid + inflow_buildings

        if key == 'Cement':
            # add concrete to cement
            inflow_buildings_cement = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material=key).loc[1961:]
            inflow_buildings_concrete = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete
            inflow_electricity_genereation =inflow_materials_generation.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete
            inflow_electricity_grid = inflow_materials_grid.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete

            total_material = inflow_electricity_genereation + inflow_electricity_grid + inflow_buildings_cement + inflow_buildings_concrete
        
        if key == 'Sand':
            inflow_buildings_cement_sand = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Cement').loc[1961:]*sand_in_cement_conversion
            inflow_buildings_concrete_sand_via_cement = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion
            inflow_buildings_sand_in_concrete = model_name.buildings.get('inflow_materials').to_array().sum(['Type']).sel(material='Concrete').loc[1961:] * sand_gravel_in_concrete_conversion
            inflow_vehicles_sand = model_name.vehicles.get('inflow_materials').to_array().sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion

            inflow_electricity_genereation_sand = inflow_materials_generation.sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion
            inflow_electricity_grid_sand = inflow_materials_grid.sum(['Type']).sel(material='Glass').loc[1961:] * sand_in_glass_conversion

            inflow_electricity_genereation_concrete_sand_via_cement = inflow_materials_generation.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion
            inflow_electricity_grid_concrete_sand_via_cement = inflow_materials_grid.sum(['Type']).sel(material='Concrete').loc[1961:] * cement_in_concrete * sand_in_cement_conversion

            total_material = (inflow_buildings_cement_sand + inflow_buildings_concrete_sand_via_cement + inflow_buildings_sand_in_concrete + 
                              inflow_vehicles_sand + inflow_electricity_genereation_sand + inflow_electricity_grid_sand + 
                              inflow_electricity_genereation_concrete_sand_via_cement + inflow_electricity_grid_concrete_sand_via_cement)

        # from total_material create a csv that has the years as rows and regions as columns, mae sure that region names are no just '1' but 'class_ 1'
        # also drop material dimension
        if key not in ['Copper', 'Cement', 'Sand']:
            total_material = total_material.drop_vars('material')
        # change the region coordinate so that it is class_ 1 instead of 1 , ...
        # Get the current region values
        regions = total_material.coords['Region'].values

        # Create new region names
        new_regions = [f'class_ {r}' for r in regions]

        # Assign the new region names to the coordinate
        total_material = total_material.assign_coords(Region=new_regions)
        # to t
        total_material = total_material.pint.to('t')
        # save as pandas to save as csv
        total_material = total_material.rename("total_material")
        # write key with a small letter
        key = key.lower()
        # to pandas
        total_material = total_material.to_dataframe().unstack()
        # drop unessecary column level index
        total_material.columns = total_material.columns.droplevel(0)
        # save as csv
        if key == 'sand':
            key = 'sand_gravel_crushed_rock'
            total_material = total_material.loc[1971:]
        else: 
            pass

        if save == True:
            total_material.to_csv(f'../data/raw/rest-of/{resource_group}/image_materials_{key}.csv')
            print('done', key)

        total_material_dict[key] = total_material

    return total_material_dict, inflow_vehicles, inflow_electricity_genereation, inflow_buildings

total_material_dict_metals, inflow_vehicles_metals, inflow_electricity_genereation_metals, inflow_buildings_metals = sum_inflows_for_output(model, materials_dict_metal, 'metals', save = True)


done steel
done aluminium
done copper


c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)
c:\Users\Arp00003\AppData\Local\miniconda3\envs\materials_dev\Lib\site-packages\xarray\core\variable.py:336: UnitStrippedWarning: The unit of the quantity is stripped when downcasting to ndarray.
  data = np.asarray(data)


In [49]:
inflow_vehicles_metals

Magnitude,[[9962411.71601334 69446838.74943331 1320032.7604785562 ... 4616646.886997723 254204.92899478265 169509.33613866108] [13375250.754253168 93132564.20705421 1753087.2480703006 ... 6204782.403167041 328019.7002075639 206296.2374658577] [16781035.15926022 116769329.22321253 2185246.574611192 ... 7789635.105976208 401681.88990910206 243007.0970888494] ... [507213671.5127944 2229032406.8058534 178635110.1604137 ... 425995453.6486685 423830560.7228278 184548539.68546668] [504759816.47139925 2230785775.8120966 184331769.06512702 ... 426850588.8876014 440472704.6508848 196127690.65363577] [501974791.6300849 2220901187.974303 189606696.76303878 ... 426743657.74797225 458255992.8686788 208445635.2422587]]
Units,kilogram


In [50]:
inflow_buildings_metals

Magnitude,[[3239411.4473689245 37302195.822067685 6801974.206781554 ... 3784629.114338422 8073646.098890849 3531177.106703311] [3405723.213780267 38717596.64853538 7080430.360607541 ... 3910315.7977782506 8387744.984051818 3661532.8283800264] [3555659.1437866944 39930261.109019175 7342932.80563549 ... 4026801.1524648857 8683325.216909254 3786628.092176444] ... [14581601.242100285 112903871.41378233 42606190.60578043 ... 11690414.000343943 147701504.00416705 65772076.11042735] [14635963.374469284 113459330.82290226 42826584.412537895 ... 11749822.80938727 151736374.29021257 68510401.96438608] [14689532.277081653 113884809.43846998 42855166.18549342 ... 11810003.57178893 152466073.15075418 69121971.17347738]]
Units,kilogram


In [54]:
inflow_electricity_genereation_metals + inflow_vehicles_metals + inflow_buildings_metals

Magnitude,[[10256042.940166408 19654520.970428675 14732832.88787454 ... 6835324.524862688 10033677.013704805 120028639.32603928] [11658334.049723882 21934495.791627843 18311993.304956734 ... 7385400.344856836 10452608.43415802 139717658.8375128] [13035998.34279498 24196324.111398973 21867732.88634603 ... 7927355.213778884 10853882.376900254 159202024.61545214] ... [1301921547.0100389 468291214.08447206 565351535.5221537 ... 163831702.27478948 1079383627.4070568 3353471091.6650524] [1340919134.7621136 468562607.75368875 570621039.27393 ... 166620342.51261202 1143932059.1156363 3499179299.5465] [1351070071.9972754 474479695.400178 575439523.6771636 ... 172536647.59290174 1206060594.399034 3612983484.3886867]]
Units,kilogram


In [29]:
total_material_dict_metals["copper"].columns

Index(['class_ 1', 'class_ 2', 'class_ 3', 'class_ 4', 'class_ 5', 'class_ 6',
       'class_ 7', 'class_ 8', 'class_ 9', 'class_ 10', 'class_ 11',
       'class_ 12', 'class_ 13', 'class_ 14', 'class_ 15', 'class_ 16',
       'class_ 17', 'class_ 18', 'class_ 19', 'class_ 20', 'class_ 21',
       'class_ 22', 'class_ 23', 'class_ 24', 'class_ 25', 'class_ 26',
       'class_ 1', 'class_ 2', 'class_ 3', 'class_ 4', 'class_ 5', 'class_ 6',
       'class_ 7', 'class_ 8', 'class_ 9', 'class_ 10', 'class_ 11',
       'class_ 12', 'class_ 13', 'class_ 14', 'class_ 15', 'class_ 16',
       'class_ 17', 'class_ 18', 'class_ 19', 'class_ 20', 'class_ 21',
       'class_ 22', 'class_ 23', 'class_ 24', 'class_ 25', 'class_ 26'],
      dtype='object', name='Region')

In [37]:
inflow_materials = model.grid["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])
inflow_materials_vehicles = model.vehicles["inflow_materials"].to_array().sel(material='Cu').sum(['Type'])

inflow_materials_vehicles


Magnitude,[[503053918.788158 3510468510.7393847 66995459.17247633 ... 231455552.5790785 13345581.05996393 9315935.988408279] [9962411.716013337 69446838.74943331 1320032.7604785562 ... 4616646.886997723 254204.92899478268 169509.33613866108] [13375250.754253168 93132564.20705421 1753087.248070301 ... 6204782.403167041 328019.700207564 206296.23746585776] ... [507213671.51279444 2229032406.805853 178635110.16041374 ... 425995453.6486685 423830560.72282785 184548539.68546668] [504759816.47139925 2230785775.812096 184331769.06512702 ... 426850588.8876013 440472704.65088475 196127690.65363574] [501974791.6300848 2220901187.974303 189606696.7630388 ... 426743657.74797225 458255992.86867875 208445635.2422587]]
Units,kilogram
